# Introduction to SQL with Python (sqlite3 and pandas.read_sql)

#### What is SQL?

SQL (Structured Query Language) is a language used to manage data in relational databases.

Think of it like Excel formulas but for databases - it lets you create, read, update, and delete data in a structured way.

Standardized in 1986; there are several different dialects


#### SQL Terms

|Term            |Description                                              |
|----------------|---------------------------------------------------------|
|Table           |Data structure which organizes data into rows and columns|
|Column          |Field in a table which stores particular attribute for records (price, name, symbol). Each column can hold a specific data type|
|Row             |A single record containing values for each column in the table|
|Query           |A request to retrieve, insert, update, or delete data from the database|



#### SQL Data Types

|SQLite Type |Description                                                                                          |Example Usage|
|------------|-----------------------------------------------------------------------------------------------------|-------------|
|INTEGER     |Whole numbers (positive, negative, or zero). Can also be used for IDs and auto-increment primary keys|coin_id INTEGER PRIMARY KEY|
|REAL        |Floating-point numbers (decimals). Good for prices, percentages, etc.|price REAL|
|TEXT        |Any string of characters. For names, symbols, descriptions, etc.|name TEXT, symbol TEXT|
|DATE/TEXT   |SQLite has no dedicated DATE type — dates are usually stored as TEXT in YYYY-MM-DD format, or as an INTEGER (Unix timestamp)|date TEXT|


#### SQL Commands

- SELECT: 
We use SELECT command to read data; probably most common command for Data Analytics (Dune queries read Dune’s DB w/ SELECT command access)

- INSERT:
Insert is used to add new records; db management

- UPDATE: 
Update is used to modify existing data

- DELETE:
Delete is used to Remove records


#### SQL Functions

|Function Type |Description                                             |Example Usage|
|--------------|--------------------------------------------------------|-------------|
|SUM           |Calculates total of a column|coin_id INTEGER PRIMARY KEY|SELECT SUM(volume) from dex.trades|
|MAX           |Returns max value in a column|price REAL|SELECT MAX(price) from prices|
|MIN           |Returns min value in a column|name TEXT, symbol TEXT|SELECT MIN(tx_count) from daily_tx|
|AVG           |Calculates the average of a column|date TEXT|SELECT AVG(users) from weekly_users|
|COUNT         |Counts number of rows in a column|SELECT COUNT(address) from ledger|


#### Real World Example

Most Blockchain Data Analysts exposed to SQL through Dune Analytics

Dune allows users to query blockchain data using SQL queries on Dune’s DB

Dune collects transaction and event data for several blockchains and allows users to create dashboards and download the data through APIs

Example Query: https://dune.com/queries/5616897?category=abstraction&namespace=dex&id=dex.trades


#### Core SQL Commands

Here are the four essential SQL commands (often called CRUD operations).

## SELECT - Read Data

SELECT * FROM coins -- Gets all the data from the coins table
SELECT name, price FROM coins WHERE price > 100; -- Get specific columns with a condition

## INSERT - Add new records

INSERT INTO coins VALUES ('Bitcoin', 50000); -- Adds new row
INSERT INTO coins VALUES ('Ethereum', 3400);

## UPDATE - Modify existing data

UPDATE coins SET price = 51000 where name = 'Bitcoin'; -- Chanes Bitcoin's price

## DELETE - Removes records

DELETE FROM coins WHERE price < 10; -- Delete all coins below 10$

# Using sqlite3 in Python

## Setting up a Database

Let's create a simple dashboard to track cryptocurrencies

In [17]:
import sqlite3
import os # This is the module that allows interaction with the file

In [20]:
db_path = "crypton.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("Database file deleted")
else:
    print ("Database file does not exist")

Database file does not exist


In [22]:
conn = sqlite3.connect(db_path)

cursor = conn.cursor()

# Create a table
cursor.execute("""
CREATE TABLE IF NOT EXISTS coins (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    symbol TEXT NOT NULL,
    price REAL,
    market_cap REAL,
    last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

conn.commit()

print ("Table 'coins' created sucessfully")

Table 'coins' created sucessfully


### Inserting Data

In [23]:
# Insert single record
cursor.execute("""
INSERT INTO coins (name, symbol, price, market_cap)
VALUES ('Bitcoin', 'BTC', 50000, 95000000000)
""")

# Insert multiple records

coins = [
    ('Ethereum', 'ETH', 3000, 350000000000),
    ('Cardano', 'ADA', 1.5, 5000000000000),
    ('Solana', "SOL", 150, 560000000000)
]

cursor.executemany ("""
INSERT INTO coins (name, symbol, price, market_cap)
VALUES (?,?,?,?)
""", coins)

conn.commit()

### Querying Data

In [24]:
cursor.execute("SELECT * FROM coins")

all_coins = cursor.fetchall()

print("All coins:", all_coins)

All coins: [(1, 'Bitcoin', 'BTC', 50000.0, 95000000000.0, '2026-03-26 18:54:50'), (2, 'Ethereum', 'ETH', 3000.0, 350000000000.0, '2026-03-26 18:54:50'), (3, 'Cardano', 'ADA', 1.5, 5000000000000.0, '2026-03-26 18:54:50'), (4, 'Solana', 'SOL', 150.0, 560000000000.0, '2026-03-26 18:54:50')]


In [26]:
cursor.execute("SELECT * FROM coins WHERE symbol = 'BTC'")

bitcoin = cursor.fetchall()

print("Bitcoin data:", bitcoin)

Bitcoin data: [(1, 'Bitcoin', 'BTC', 50000.0, 95000000000.0, '2026-03-26 18:54:50')]


### Updating and Deleting

In [30]:
cursor.execute("UPDATE coins SET price = 51000 WHERE symbol = 'BTC'")

conn.commit()

print("Bitcoin updated")

Bitcoin updated


In [31]:
cursor.execute("DELETE FROM coins WHERE price < 2")

conn.commit()

print ("Prices less than 2 deleted")

Prices less than 2 deleted


### Using Context Managers (Best Practice)

In [33]:
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM coins")
    print(cursor.fetchall())

[(1, 'Bitcoin', 'BTC', 51000.0, 95000000000.0, '2026-03-26 18:54:50'), (2, 'Ethereum', 'ETH', 3000.0, 350000000000.0, '2026-03-26 18:54:50'), (4, 'Solana', 'SOL', 150.0, 560000000000.0, '2026-03-26 18:54:50')]


### Pandas + SQL Integration
Pandas makes working with SQL data even easier by converting between DataFrames and SQL tables

## Reading SQL into Pandas

In [34]:
! pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql("SELECT * FROM coins", conn)

print("DataFrame from SQL:")
print(df)

DataFrame from SQL:
   id      name symbol    price    market_cap         last_updated
0   1   Bitcoin    BTC  51000.0  9.500000e+10  2026-03-26 18:54:50
1   2  Ethereum    ETH   3000.0  3.500000e+11  2026-03-26 18:54:50
2   4    Solana    SOL    150.0  5.600000e+11  2026-03-26 18:54:50


### Writing Pandas into SQL

In [43]:
new_coins = pd.DataFrame({
    'name': ['Polkadot', 'Polygon'],
    'symbol': ['DOT', 'MATIC'],
    'price': [7.5, 0.75],
    'market_cap': [750000000000, 500000000000]
})

with sqlite3.connect(db_path) as conn:
    new_coins.to_sql("new_coins", conn, index=False, if_exists='replace')

    df_new = pd.read_sql("SELECT * FROM new_coins", conn)
print(df_new)      

       name symbol  price    market_cap
0  Polkadot    DOT   7.50  750000000000
1   Polygon  MATIC   0.75  500000000000


### Analyzing SQL Data with Pandas

In [51]:
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql("SELECT * FROM coins", conn)

print("\nBasic statistics:")
print(df.describe())


Basic statistics:
             id         price    market_cap
count  3.000000      3.000000  3.000000e+00
mean   2.333333  18050.000000  3.350000e+11
std    1.527525  28571.095534  2.328626e+11
min    1.000000    150.000000  9.500000e+10
25%    1.500000   1575.000000  2.225000e+11
50%    2.000000   3000.000000  3.500000e+11
75%    3.000000  27000.000000  4.550000e+11
max    4.000000  51000.000000  5.600000e+11


In [52]:

print("\nAverage price:", df['price'].mean())


Average price: 18050.0


In [53]:
print("Total market cap:", df['market_cap'].sum())


Total market cap: 1005000000000.0


In [54]:
print("\nExpensive coins:")
print(df[['name', 'price']])


Expensive coins:
       name    price
0   Bitcoin  51000.0
1  Ethereum   3000.0
2    Solana    150.0


### JOINS & Relationships

#### Understanding Table Relationships

In real databases, data is often split across multiple related tables. For example:
- coins table: coin_id, name, symbol
- prices table: price_id, coin_id, date

#### Creating Related Tables

In [58]:
with sqlite3.connect(db_path) as conn:

    cursor = conn.cursor()

    # Create coins table
    cursor.execute("DROP TABLE IF EXISTS coins")
    cursor.execute("""
    CREATE TABLE coins (
        coin_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        symbol TEXT NOT NULL UNIQUE
    )
    """)

    # Create prices table with foreign key
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS prices (
        price_id INTEGER PRIMARY KEY AUTOINCREMENT,
        coin_id INTEGER NOT NULL,
        price REAL NOT NULL,
        date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (coin_id) REFERENCES coins (coin_id)
    )
    """)

    # Insert some data
    cursor.execute("INSERT INTO coins (name, symbol) VALUES ('Bitcoin', 'BTC')")
    cursor.execute("INSERT INTO coins (name, symbol) VALUES ('Ethereum', 'ETH')")

    # Get the auto-generated IDs
    eth_id = cursor.lastrowid # This would actually get ETH's ID - in real code you'd need to query for BTC's ID
    btc_id = eth_id - 1

    # Insert price history
    cursor.executemany("""
    INSERT INTO prices (coin_id, price)
    VALUES (?, ?)
    """, [
        (btc_id, 50000),
        (btc_id, 51000),
        (eth_id, 3000),
        (eth_id, 3100)
    ])

    conn.commit()

#### Performing JOINS

- Inner join: keeps only rows that match in both tables

- Left Join: **Keeps all rows from the left table**, and matches from the right table if available.

- Right Join: **Keeps all rows from the right table**, matches from the left if available.

- Full Outer Join: **Keeps all rows from both tables**, matching where possible.

In [59]:
with sqlite3.connect(db_path) as conn:
    # Simple inner join
    df = pd.read_sql("""
    SELECT c.name, c.symbol, p.price, p.date
    FROM coins c
    INNER JOIN prices p ON c.coin_id = p.coin_id
    ORDER BY p.date DESC
    """, conn)

print("\nCoin prices with join:")
print(df)


Coin prices with join:
        name symbol    price                 date
0    Bitcoin    BTC  50000.0  2026-03-27 18:04:40
1    Bitcoin    BTC  51000.0  2026-03-27 18:04:40
2   Ethereum    ETH   3000.0  2026-03-27 18:04:40
3   Ethereum    ETH   3100.0  2026-03-27 18:04:40
4    Bitcoin    BTC  50000.0  2026-03-27 17:51:21
5    Bitcoin    BTC  51000.0  2026-03-27 17:51:21
6   Ethereum    ETH   3000.0  2026-03-27 17:51:21
7   Ethereum    ETH   3100.0  2026-03-27 17:51:21
8    Bitcoin    BTC  50000.0  2026-03-27 13:29:29
9    Bitcoin    BTC  51000.0  2026-03-27 13:29:29
10  Ethereum    ETH   3000.0  2026-03-27 13:29:29
11  Ethereum    ETH   3100.0  2026-03-27 13:29:29


In [61]:
# Example of LEFT JOIN (get all coins even if they have prices)
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql("""
    SELECT c.name, c.symbol, p.price
    FROM coins c
    LEFT JOIN prices p ON c.coin_id = p.coin_id
    """, conn)

print("\nAll coins with prices (if available):")
print(df)


All coins with prices (if available):
        name symbol    price
0    Bitcoin    BTC  50000.0
1    Bitcoin    BTC  50000.0
2    Bitcoin    BTC  50000.0
3    Bitcoin    BTC  51000.0
4    Bitcoin    BTC  51000.0
5    Bitcoin    BTC  51000.0
6   Ethereum    ETH   3000.0
7   Ethereum    ETH   3000.0
8   Ethereum    ETH   3000.0
9   Ethereum    ETH   3100.0
10  Ethereum    ETH   3100.0
11  Ethereum    ETH   3100.0


### Best Practices

In [ ]:
# Use context managers (with statement - you dont need to use conn commit())

with sqlite3.connect(db_path) as conn:
    df = pd.read_sql("""
    SELECT *
    FROM coins c
    """, conn)

In [ ]:
# Close connection with commit (if not using context manager)

conn = sqlite3.connect(db_path)
df = pd.read_sql("""
SELECT *
FROM coins
""", conn)
conn.commit() # saving
conn.close() # closing the database connection